In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (reuse variable names)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Helper to build a compact summary for a file-like DataFrame
def build_summary(df, name_col=None, message_prefix=False):
    row_count = len(df)
    cmdb_ids = sorted(df['cmdb_id'].dropna().unique().tolist())
    if message_prefix:
        # create message prefixes (clean newlines, take up to 120 chars)
        prefixes = (
            df['message'].astype(str)
              .str.replace(r'\s+', ' ', regex=True)
              .str.slice(0, 120)
        )
        name_samples = sorted(prefixes.dropna().unique().tolist())[:20]
    elif name_col and name_col in df.columns:
        name_samples = sorted(df[name_col].dropna().unique().tolist())[:20]
    else:
        name_samples = []
    # min/max timestamps in ISO format (UTC-aware)
    if row_count:
        min_ts = df['timestamp'].min().isoformat()
        max_ts = df['timestamp'].max().isoformat()
    else:
        min_ts = max_ts = None
    # Compose compact summary as a one-row DataFrame
    summary = pd.DataFrame([{
        'rows': row_count,
        'cmdb_ids': cmdb_ids,
        'name_samples(up to20)': name_samples,
        'min_timestamp_utc': min_ts,
        'max_timestamp_utc': max_ts
    }])
    return summary

# Build summaries
metric_summary = build_summary(metric_df, name_col='kpi_name')
trace_summary = build_summary(trace_df, name_col='trace_name')
log_summary = build_summary(log_df, name_col='log_name')
error_summary = build_summary(error_df, message_prefix=True)

# Top 5 rows for each (keep original columns)
metric_top5 = metric_df.head(5)
trace_top5 = trace_df.head(5)
log_top5 = log_df.head(5)
error_top5 = error_df.head(5)

# Display compact summaries and top5 tables (IPython will render them)
metric_summary, metric_top5, trace_summary, trace_top5, log_summary, log_top5, error_summary, error_top5

```
Out[1]:
```
# Compose a concise plain-English summary based on previously computed summaries
metric_info = metric_summary.iloc[0]
trace_info = trace_summary.iloc[0]
log_info = log_summary.iloc[0]
error_info = error_summary.iloc[0]

def sample_list(lst, n=10):
    if not lst:
        return "[]"
    s = lst[:n]
    return "[" + ", ".join(s) + (", ..." if len(lst) > n else "") + "]"

summary = (
    "Summary of loaded telemetry files (compact):\n\n"
    "metric.csv:\n"
    f"- Total rows: {int(metric_info['rows'])}\n"
    f"- Unique cmdb_id sample (first up to 10): {sample_list(metric_info['cmdb_ids'], 10)}\n"
    f"- KPI name samples (up to 10 of the reported list): {sample_list(metric_info['name_samples(up to20)'], 10)}\n"
    f"- Time range (UTC): {metric_info['min_timestamp_utc']} to {metric_info['max_timestamp_utc']}\n"
    "- Top 5 rows: shown above to confirm contents.\n\n"
    "trace.csv:\n"
    f"- Total rows: {int(trace_info['rows'])}\n"
    f"- Unique cmdb_id sample (first up to 10): {sample_list(trace_info['cmdb_ids'], 10)}\n"
    f"- Trace name samples (up to 10 of the reported list): {sample_list(trace_info['name_samples(up to20)'], 10)}\n"
    f"- Time range (UTC): {trace_info['min_timestamp_utc']} to {trace_info['max_timestamp_utc']}\n"
    "- Top 5 rows: shown above to confirm contents.\n\n"
    "log.csv:\n"
    f"- Total rows: {int(log_info['rows'])}\n"
    f"- Unique cmdb_id sample (first up to 10): {sample_list(log_info['cmdb_ids'], 10)}\n"
    f"- Log name samples (up to 10): {sample_list(log_info['name_samples(up to20)'], 10)}\n"
    f"- Time range (UTC): {log_info['min_timestamp_utc']} to {log_info['max_timestamp_utc']}\n"
    "- Top 5 rows: shown above to confirm contents.\n\n"
    "error_logs.csv:\n"
    f"- Total rows: {int(error_info['rows'])}\n"
    "- No cmdb_ids or message prefixes (file is empty in this dataset).\n\n"
    "Notes:\n"
    "- All timestamps are parsed as UTC. The three populated files share the same timestamp window: 2024-01-23T21:04:00+00:00 through 2024-01-23T21:28:00+00:00.\n"
    "- Top 5 rows for each file were displayed previously for verification."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows                                           cmdb_ids                              name_samples(up to20)          min_timestamp_utc          max_timestamp_utc
0  8948  [ts-admin-basic-info-service, ts-admin-order-s...  [cpu, diskio, error, latency-50, latency-90, m...  2024-01-23T21:04:00+00:00  2024-01-23T21:28:00+00:00,                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service         cpu      0.460474
1 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service      diskio  53827.946152
2 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service       error      0.000000
3 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service  latency-50      0.017500
4 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service  latency-90      0.023500,     rows                                           cmdb_ids                              name_samples(up to20)          min_timestamp_utc          max_timestamp_utc
0  18632  [root, ts-admin-basic-info-service, ts-admin-t...  [trace.from_root.duration_mean, trace.from_roo...  2024-01-23T21:04:00+00:00  2024-01-23T21:28:00+00:00,                   timestamp cmdb_id                                         trace_name      value
0 2024-01-23 21:04:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_...   0.012076
1 2024-01-23 21:04:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_p95   0.014861
2 2024-01-23 21:04:00+00:00    root    trace.to_ts-admin-basic-info-service.error_rate   0.000000
3 2024-01-23 21:04:00+00:00    root     trace.to_ts-admin-basic-info-service.row_count  11.000000
4 2024-01-23 21:04:00+00:00    root     trace.to_ts-admin-travel-service.duration_mean   0.143295,    rows                                           cmdb_ids             name_samples(up to20)          min_timestamp_utc          max_timestamp_utc
0  2194  [ts-admin-basic-info-service, ts-admin-travel-...  [log.error_count, log.row_count]  2024-01-23T21:04:00+00:00  2024-01-23T21:28:00+00:00,                   timestamp                      cmdb_id         log_name  value
0 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service  log.error_count      0
1 2024-01-23 21:04:00+00:00  ts-admin-basic-info-service    log.row_count     16
2 2024-01-23 21:04:00+00:00      ts-admin-travel-service  log.error_count      0
3 2024-01-23 21:04:00+00:00      ts-admin-travel-service    log.row_count     12
4 2024-01-23 21:04:00+00:00         ts-assurance-service  log.error_count      0,    rows cmdb_ids name_samples(up to20) min_timestamp_utc max_timestamp_utc
0     0       []                    []              None              None, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric_df if present; otherwise load and parse
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Determine which candidates appear in the file
present = sorted(set(metric_df['cmdb_id'].dropna().unique()).intersection(candidates))

if not present:
    result = "Empty result: none of the candidate components are present in metric.csv"
else:
    df_cand = metric_df[metric_df['cmdb_id'].isin(present)].copy()
    df_cand['value'] = pd.to_numeric(df_cand['value'], errors='coerce')

    # Compute full_count and global P95 per group BEFORE window filtering
    group_stats = (
        df_cand
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(full_count='size', p95_threshold=lambda s: float(s.quantile(0.95)))
    )

    # Incident window (UTC)
    window_start = pd.to_datetime('2024-01-23 21:01:34', utc=True)
    window_end   = pd.to_datetime('2024-01-23 21:31:34', utc=True)

    # Rows within the window
    window_df = df_cand[(df_cand['timestamp'] >= window_start) & (df_cand['timestamp'] <= window_end)].copy()

    if window_df.empty:
        group_stats['window_count'] = 0
        group_stats['window_anomaly_count'] = 0
        group_stats['earliest_anomaly_timestamp_utc'] = None
        group_stats['max_value_in_window'] = pd.NA
        result_df = group_stats[['cmdb_id','kpi_name','full_count','p95_threshold',
                                 'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']]
    else:
        # Attach p95 to window rows and flag anomalies
        window_merged = window_df.merge(group_stats[['cmdb_id','kpi_name','p95_threshold']],
                                        on=['cmdb_id','kpi_name'], how='left')
        window_merged['is_anomaly'] = window_merged['value'] > window_merged['p95_threshold']

        # Aggregates in window
        window_counts = (
            window_merged
            .groupby(['cmdb_id','kpi_name'], as_index=False)
            .agg(window_count=('value','size'),
                 window_anomaly_count=('is_anomaly','sum'),
                 max_value_in_window=('value','max'))
        )

        # Earliest anomaly timestamp per group (if any)
        anomalies = window_merged[window_merged['is_anomaly']]
        if not anomalies.empty:
            earliest_anom = (
                anomalies
                .groupby(['cmdb_id','kpi_name'], as_index=False)
                ['timestamp'].min()
            )
            # Format timestamps to ISO-like strings
            earliest_anom['earliest_anomaly_timestamp_utc'] = earliest_anom['timestamp'].dt.tz_convert('UTC').astype(str)
            earliest_anom = earliest_anom.drop(columns=['timestamp'])
        else:
            earliest_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp_utc'])

        # Combine stats
        result_df = group_stats.merge(window_counts, on=['cmdb_id','kpi_name'], how='left')
        result_df = result_df.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

        # Fill missing window aggregates
        result_df['window_count'] = result_df['window_count'].fillna(0).astype(int)
        result_df['window_anomaly_count'] = result_df['window_anomaly_count'].fillna(0).astype(int)
        result_df['max_value_in_window'] = result_df['max_value_in_window'].where(result_df['max_value_in_window'].notna(), pd.NA)
        result_df['earliest_anomaly_timestamp_utc'] = result_df['earliest_anomaly_timestamp_utc'].where(
            result_df['earliest_anomaly_timestamp_utc'].notna(), None
        )

        result_df = result_df[['cmdb_id','kpi_name','full_count','p95_threshold',
                               'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']]

    result = result_df.head(100)

result

```
Out[2]:
```
Summary (plain English):

- Candidate services present in metric.csv: all five—ts-auth-service, ts-order-service, ts-route-service, ts-train-service, and ts-travel-service—appear in the file and were analyzed.

- Aggregate finding per component-KPI:
  - Each group has 25 total points in the full series (full_count = 25).
  - A global P95 was computed per group; during the incident window every group had data (window_count = 25).
  - For every reported group, there were a small number of values above the group’s global P95 (window_anomaly_count = 2 for each KPI group shown).
  - Earliest anomaly timestamps occur within the incident window; many KPIs show their first exceedance at or shortly after 21:04:00 UTC (examples below).
  - Max values in-window exceed the P95 thresholds (examples below).

- Examples (selected groups and their earliest anomaly times / max values):
  - ts-auth-service:
    - cpu: earliest anomaly 2024-01-23T21:08:00Z, max ≈ 16.56, P95 ≈ 15.66
    - diskio: earliest anomaly 2024-01-23T21:04:00Z, max ≈ 138450.1, P95 ≈ 100999.6
    - latency-50 / latency-90: earliest anomalies ~21:20:00Z, maxes slightly above P95
    - mem: earliest anomaly 21:13:00Z, max slightly above P95
  - ts-travel-service:
    - latency-50 / latency-90: earliest anomaly 21:04:00Z, maxs above P95
    - mem, socket, workload: earliest anomalies between 21:04 and 21:21 UTC, maxs above P95

- Interpretation:
  - Multiple KPIs across all candidate services exceeded their historical (global) P95 during the incident window, indicating short-lived but repeated spikes rather than a single sustained exceedance (each group shows 2 samples above P95 out of 25).
  - Earliest exceedances commonly start around 21:04:00 UTC, well inside the requested incident window (21:01:34–21:31:34 UTC).

- Recommended next steps:
  - Inspect traces and logs for the same services/time range (especially around 21:04–21:13 UTC and ~21:20 UTC) to determine if latency/diskio/memory spikes correlate with errors or trace-level failures.
  - Prioritize investigation of disk I/O and memory and latency metrics (they show clear exceedance of P95) for ts-auth-service and ts-travel-service as starting points.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  full_count  p95_threshold  window_count  window_anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window
0     ts-auth-service         cpu          25   1.566377e+01            25                     2      2024-01-23 21:08:00+00:00         1.656489e+01
1     ts-auth-service      diskio          25   1.009996e+05            25                     2      2024-01-23 21:04:00+00:00         1.384501e+05
2     ts-auth-service  latency-50          25   3.311969e-01            25                     2      2024-01-23 21:20:00+00:00         3.405400e-01
3     ts-auth-service  latency-90          25   8.879521e-01            25                     2      2024-01-23 21:20:00+00:00         9.028121e-01
4     ts-auth-service         mem          25   2.498109e+08            25                     2      2024-01-23 21:13:00+00:00         2.508810e+08
..                ...         ...         ...            ...           ...                   ...                            ...                  ...
30  ts-travel-service  latency-50          25   1.665142e-02            25                     2      2024-01-23 21:04:00+00:00         1.726229e-02
31  ts-travel-service  latency-90          25   2.508579e-01            25                     2      2024-01-23 21:04:00+00:00         2.752396e-01
32  ts-travel-service         mem          25   2.650515e+08            25                     2      2024-01-23 21:04:00+00:00         2.655178e+08
33  ts-travel-service      socket          25   1.852667e+01            25                     2      2024-01-23 21:11:00+00:00         1.956667e+01
34  ts-travel-service    workload          25   1.077793e+01            25                     2      2024-01-23 21:21:00+00:00         1.122106e+01

[35 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate services present in metrics (derive from metric_df if available)
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
try:
    present_services = sorted(set(metric_df['cmdb_id'].dropna().unique()).intersection(candidates))
except NameError:
    # Fallback: intersect with candidates that appear in trace file
    present_services = sorted(set(trace_df['cmdb_id'].dropna().unique()).intersection(candidates))

if not present_services:
    result_trace = "Empty result: none of the candidate services have trace rows in trace.csv"
else:
    # Filter traces to only those candidate services present
    traces_cand = trace_df[trace_df['cmdb_id'].isin(present_services)].copy()

    # Compute full-series stats per group BEFORE window filtering
    group_stats = (
        traces_cand
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(full_count='size', p95_threshold=lambda s: float(s.quantile(0.95)))
    )

    # Incident window (UTC)
    window_start = pd.to_datetime('2024-01-23 21:01:34', utc=True)
    window_end   = pd.to_datetime('2024-01-23 21:31:34', utc=True)

    # Rows within the window
    window_df = traces_cand[(traces_cand['timestamp'] >= window_start) & (traces_cand['timestamp'] <= window_end)].copy()

    if window_df.empty:
        # No data in window for any group
        group_stats['window_count'] = 0
        group_stats['window_anomaly_count'] = 0
        group_stats['earliest_anomaly_timestamp_utc'] = None
        group_stats['max_value_in_window'] = pd.NA
        result_df = group_stats[['cmdb_id','trace_name','full_count','p95_threshold',
                                 'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']]
        result_trace = result_df.head(100)
    else:
        # Attach p95 to window rows and flag anomalies
        window_merged = window_df.merge(group_stats[['cmdb_id','trace_name','p95_threshold']],
                                        on=['cmdb_id','trace_name'], how='left')
        window_merged['is_anomaly'] = window_merged['value'] > window_merged['p95_threshold']

        # Aggregates in window
        window_counts = (
            window_merged
            .groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(window_count=('value','size'),
                 window_anomaly_count=('is_anomaly','sum'),
                 max_value_in_window=('value','max'))
        )

        # Earliest anomaly timestamp per group (if any)
        anomalies = window_merged[window_merged['is_anomaly']]
        if not anomalies.empty:
            earliest_anom = (
                anomalies
                .groupby(['cmdb_id','trace_name'], as_index=False)
                ['timestamp'].min()
            )
            earliest_anom['earliest_anomaly_timestamp_utc'] = earliest_anom['timestamp'].dt.tz_convert('UTC').astype(str)
            earliest_anom = earliest_anom.drop(columns=['timestamp'])
        else:
            earliest_anom = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp_utc'])

        # Combine stats
        result_df = group_stats.merge(window_counts, on=['cmdb_id','trace_name'], how='left')
        result_df = result_df.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

        # Fill missing window aggregates
        result_df['window_count'] = result_df['window_count'].fillna(0).astype(int)
        result_df['window_anomaly_count'] = result_df['window_anomaly_count'].fillna(0).astype(int)
        result_df['max_value_in_window'] = result_df['max_value_in_window'].where(result_df['max_value_in_window'].notna(), pd.NA)
        result_df['earliest_anomaly_timestamp_utc'] = result_df['earliest_anomaly_timestamp_utc'].where(
            result_df['earliest_anomaly_timestamp_utc'].notna(), None
        )

        result_df = result_df[['cmdb_id','trace_name','full_count','p95_threshold',
                               'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']]

        result_trace = result_df.head(100)

# Display compact result variable
result_trace

```
Out[3]:
```
Summary (plain English):

- Scope: traces were restricted to the five candidate services and analyzed across the full series and the incident window 2024-01-23 21:01:34–21:31:34 UTC.

- Overall pattern:
  - Each trace group has 25 data points in the full series (full_count = 25) and 25 points inside the incident window (window_count = 25).
  - Many trace metrics show a small number of spikes above their global P95 during the window (most groups show window_anomaly_count = 2).
  - Traces that measure error_rate generally have p95 = 0 and window_anomaly_count = 0 (no error-rate spikes).
  - The earliest anomaly times vary by trace; examples:
    - ts-auth-service: trace.from_root.duration_mean and duration_p95 show earliest anomaly at ~2024-01-23T21:20:00Z; row_count earliest anomaly ~21:09:00Z.
    - ts-travel-service: several traces (duration, p95, row_count from various callers) show earliest anomalies around 21:04:00–21:05:00Z.
  - Max values in-window exceed the computed P95 thresholds in the affected traces.

- Interpretation:
  - Multiple services (notably ts-auth-service and ts-travel-service among others) experienced short-lived spikes in trace durations and row_counts during the incident window, rather than sustained high error rates.
  - The spikes are small in count (typically 2 samples above P95 per trace) but present across many trace metrics, indicating intermittent latency/throughput anomalies.

- Recommended next steps:
  1. Prioritize traces and logs for ts-auth-service and ts-travel-service around the earliest anomaly timestamps (around 21:04–21:09 and ~21:20 UTC) to find causal traces or correlated errors.
  2. Correlate these trace spikes with the metric anomalies (CPU, memory, diskio, latency) already observed for the same services.
  3. If available, inspect application logs and span details at the spike timestamps to find slow operations, blocking I/O, or downstream service delays.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  full_count  p95_threshold  window_count  window_anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window
0     ts-auth-service                 trace.from_root.duration_mean          25       0.415821            25                     2      2024-01-23 21:20:00+00:00             0.426565
1     ts-auth-service                  trace.from_root.duration_p95          25       1.208624            25                     2      2024-01-23 21:20:00+00:00             1.212228
2     ts-auth-service                    trace.from_root.error_rate          25       0.000000            25                     0                           None             0.000000
3     ts-auth-service                     trace.from_root.row_count          25      88.600000            25                     2      2024-01-23 21:09:00+00:00            92.000000
4     ts-auth-service      trace.from_ts-auth-service.duration_mean          25       0.033327            25                     2      2024-01-23 21:25:00+00:00             0.035695
..                ...                                           ...         ...            ...           ...                   ...                            ...                  ...
95  ts-travel-service  trace.from_ts-admin-travel-service.row_count          25      11.800000            25                     2      2024-01-23 21:05:00+00:00            12.000000
96  ts-travel-service      trace.from_ts-food-service.duration_mean          25       0.010063            25                     2      2024-01-23 21:04:00+00:00             0.010582
97  ts-travel-service       trace.from_ts-food-service.duration_p95          25       0.015010            25                     2      2024-01-23 21:18:00+00:00             0.015600
98  ts-travel-service         trace.from_ts-food-service.error_rate          25       0.000000            25                     0                           None             0.000000
99  ts-travel-service          trace.from_ts-food-service.row_count          25      25.800000            25                     2      2024-01-23 21:19:00+00:00            29.000000

[100 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse previously loaded dataframes when available, otherwise load fresh
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (idempotent)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# --- Part 1: log.csv analysis for candidate services ---
# Filter to candidate services
logs_cand = log_df[log_df['cmdb_id'].isin(candidates)].copy()
logs_cand['value'] = pd.to_numeric(logs_cand['value'], errors='coerce')

if logs_cand.empty:
    result_log = pd.DataFrame([])  # explicit empty DataFrame result
else:
    # Compute full-series stats per group BEFORE window filtering
    group_stats = (
        logs_cand
        .groupby(['cmdb_id','log_name'], as_index=False)['value']
        .agg(full_count='size', p95_threshold=lambda s: float(s.quantile(0.95)))
    )

    # Incident window (UTC)
    window_start = pd.to_datetime('2024-01-23 21:01:34', utc=True)
    window_end   = pd.to_datetime('2024-01-23 21:31:34', utc=True)

    # Rows within the window
    window_df = logs_cand[(logs_cand['timestamp'] >= window_start) & (logs_cand['timestamp'] <= window_end)].copy()

    if window_df.empty:
        group_stats['window_count'] = 0
        group_stats['window_anomaly_count'] = 0
        group_stats['earliest_anomaly_timestamp_utc'] = None
        group_stats['max_value_in_window'] = pd.NA
        result_log = group_stats[['cmdb_id','log_name','full_count','p95_threshold',
                                  'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']]
    else:
        # Merge p95 into window rows and flag anomalies
        window_merged = window_df.merge(group_stats[['cmdb_id','log_name','p95_threshold']],
                                        on=['cmdb_id','log_name'], how='left')
        window_merged['is_anomaly'] = window_merged['value'] > window_merged['p95_threshold']

        # Aggregates in window
        window_counts = (
            window_merged
            .groupby(['cmdb_id','log_name'], as_index=False)
            .agg(window_count=('value','size'),
                 window_anomaly_count=('is_anomaly','sum'),
                 max_value_in_window=('value','max'))
        )

        # Earliest anomaly timestamp per group (if any)
        anomalies = window_merged[window_merged['is_anomaly']]
        if not anomalies.empty:
            earliest_anom = (
                anomalies
                .groupby(['cmdb_id','log_name'], as_index=False)['timestamp'].min()
            )
            earliest_anom['earliest_anomaly_timestamp_utc'] = earliest_anom['timestamp'].dt.tz_convert('UTC').astype(str)
            earliest_anom = earliest_anom.drop(columns=['timestamp'])
        else:
            earliest_anom = pd.DataFrame(columns=['cmdb_id','log_name','earliest_anomaly_timestamp_utc'])

        # Combine stats
        result_log = group_stats.merge(window_counts, on=['cmdb_id','log_name'], how='left')
        result_log = result_log.merge(earliest_anom, on=['cmdb_id','log_name'], how='left')

        # Fill missing window aggregates
        result_log['window_count'] = result_log['window_count'].fillna(0).astype(int)
        result_log['window_anomaly_count'] = result_log['window_anomaly_count'].fillna(0).astype(int)
        result_log['max_value_in_window'] = result_log['max_value_in_window'].where(result_log['max_value_in_window'].notna(), pd.NA)
        result_log['earliest_anomaly_timestamp_utc'] = result_log['earliest_anomaly_timestamp_utc'].where(
            result_log['earliest_anomaly_timestamp_utc'].notna(), None
        )

        result_log = result_log[['cmdb_id','log_name','full_count','p95_threshold',
                                 'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']]

    # Limit to top 100 rows for compactness
    result_log = result_log.head(100)

# --- Part 2: error_logs.csv overview and window rows for candidate services ---
error_total_rows = len(error_df)
error_cmdbs = sorted(error_df['cmdb_id'].dropna().unique().tolist())

# Filter error rows to candidate services within the incident window
err_window = error_df[
    (error_df['cmdb_id'].isin(candidates)) &
    (error_df['timestamp'] >= window_start) &
    (error_df['timestamp'] <= window_end)
].copy()

if err_window.empty:
    error_window_top20 = pd.DataFrame([])  # explicit empty DataFrame
    error_overview = pd.DataFrame([{
        'total_rows': error_total_rows,
        'distinct_cmdb_ids': error_cmdbs,
        'message': 'No error_logs rows for candidate services within the incident window'
    }])
else:
    # Limit to top 20 rows and show only requested columns
    error_window_top20 = err_window[['timestamp','cmdb_id','message']].sort_values('timestamp').head(20)
    error_overview = pd.DataFrame([{
        'total_rows': error_total_rows,
        'distinct_cmdb_ids': error_cmdbs,
        'rows_in_window_for_candidates': len(err_window)
    }])

# Display compact results (IPython will render these variables)
result_log, error_overview, error_window_top20

```
Out[4]:
```
Summary (plain English):

- Logs overview for candidate services:
  - log.error_count: p95 = 0 for all services; no anomalies in the incident window (window_anomaly_count = 0).
  - log.row_count: each reported service (ts-auth-service, ts-order-service, ts-route-service, ts-travel-service) has full_count = 25, p95 > 0, and 2 samples above the global P95 during the incident window (window_anomaly_count = 2 for each).
    - ts-auth-service: earliest exceedance 2024-01-23T21:09:00Z, max row_count in window = 368 (p95 ≈ 356.4).
    - ts-order-service: earliest exceedance 2024-01-23T21:08:00Z, max = 488 (p95 ≈ 437.2).
    - ts-route-service: earliest exceedance 2024-01-23T21:08:00Z, max = 1127 (p95 ≈ 1115.8).
    - ts-travel-service: earliest exceedance 2024-01-23T21:08:00Z, max = 2178 (p95 ≈ 1956.0).
  - ts-train-service did not appear in the log groups shown.

- error_logs.csv:
  - No error_log rows present for the candidate services in the dataset and none within the incident window.

- Interpretation and recommended next steps:
  - There were short-lived spikes in row_count (throughput) across multiple services around ~21:08–21:09 UTC, but no corresponding increases in error counts or stored error log messages.
  - Correlate these row_count spikes with the metric and trace anomalies already observed (CPU, diskio, memory, latency, and trace duration spikes around 21:04–21:20 UTC).
  - Inspect traces and application logs at the listed earliest anomaly timestamps (21:08–21:09 and nearby) to determine whether higher request load or downstream latency caused the metric/trace spikes. Focus on ts-auth, ts-order, ts-route, and ts-travel as starting points.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  full_count  p95_threshold  window_count  window_anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window
0    ts-auth-service  log.error_count          25            0.0            25                     0                           None                    0
1    ts-auth-service    log.row_count          25          356.4            25                     2      2024-01-23 21:09:00+00:00                  368
2   ts-order-service  log.error_count          25            0.0            25                     0                           None                    0
3   ts-order-service    log.row_count          25          437.2            25                     2      2024-01-23 21:08:00+00:00                  488
4   ts-route-service  log.error_count          25            0.0            25                     0                           None                    0
5   ts-route-service    log.row_count          25         1115.8            25                     2      2024-01-23 21:08:00+00:00                 1127
6  ts-travel-service  log.error_count          25            0.0            25                     0                           None                    0
7  ts-travel-service    log.row_count          25         1956.0            25                     2      2024-01-23 21:08:00+00:00                 2178,    total_rows distinct_cmdb_ids                                            message
0           0                []  No error_logs rows for candidate services with..., Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd, re

# Reuse existing trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Parse timestamp as UTC (idempotent) and ensure numeric value
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate substrings to search for in trace_name (match case)
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Build regex pattern to match any candidate substring (escape them)
pattern = "|".join(re.escape(s) for s in candidates)

# Filter rows where trace_name contains any candidate substring
matched = trace_df[trace_df['trace_name'].astype(str).str.contains(pattern, regex=True, na=False)].copy()

# Incident window (UTC)
window_start = pd.to_datetime('2024-01-23 21:01:34', utc=True)
window_end   = pd.to_datetime('2024-01-23 21:31:34', utc=True)

if matched.empty:
    result = "Empty result: no trace rows have trace_name containing the candidate service substrings."
else:
    # Compute full-series stats per group BEFORE window filtering
    group_stats = (
        matched
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(full_count='size', p95_threshold=lambda s: float(s.quantile(0.95)))
    )

    # Filter matched rows to the incident window
    window_df = matched[(matched['timestamp'] >= window_start) & (matched['timestamp'] <= window_end)].copy()

    if window_df.empty:
        result = "Empty result: matched trace groups exist but no rows fall inside the incident window."
    else:
        # Merge p95 into window rows and flag anomalies
        window_merged = window_df.merge(group_stats[['cmdb_id','trace_name','p95_threshold']],
                                        on=['cmdb_id','trace_name'], how='left')
        window_merged['is_anomaly'] = window_merged['value'] > window_merged['p95_threshold']

        # Aggregates in window
        window_counts = (
            window_merged
            .groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(window_count=('value','size'),
                 window_anomaly_count=('is_anomaly','sum'),
                 max_value_in_window=('value','max'))
        )

        # Earliest anomaly timestamp per group (if any)
        anomalies = window_merged[window_merged['is_anomaly']]
        if not anomalies.empty:
            earliest_anom = (
                anomalies
                .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min()
            )
            earliest_anom['earliest_anomaly_timestamp_utc'] = earliest_anom['timestamp'].dt.tz_convert('UTC').astype(str)
            earliest_anom = earliest_anom.drop(columns=['timestamp'])
        else:
            earliest_anom = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp_utc'])

        # Combine stats
        result_df = group_stats.merge(window_counts, on=['cmdb_id','trace_name'], how='left')
        result_df = result_df.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

        # Fill missing window aggregates with zeros/NA
        result_df['window_count'] = result_df['window_count'].fillna(0).astype(int)
        result_df['window_anomaly_count'] = result_df['window_anomaly_count'].fillna(0).astype(int)
        result_df['max_value_in_window'] = result_df['max_value_in_window'].where(result_df['max_value_in_window'].notna(), pd.NA)
        result_df['earliest_anomaly_timestamp_utc'] = result_df['earliest_anomaly_timestamp_utc'].where(
            result_df['earliest_anomaly_timestamp_utc'].notna(), None
        )

        # Keep only rows with anomalies in the window
        anomalies_only = result_df[result_df['window_anomaly_count'] > 0].copy()

        if anomalies_only.empty:
            result = "Empty result: matched trace groups have no anomalies (value > global P95) in the incident window."
        else:
            # Sort by earliest anomaly timestamp ascending and limit to top 20
            anomalies_only = anomalies_only.sort_values('earliest_anomaly_timestamp_utc', na_position='last')
            result = anomalies_only[['cmdb_id','trace_name','full_count','p95_threshold',
                                     'window_count','window_anomaly_count','earliest_anomaly_timestamp_utc','max_value_in_window']].head(20)

# Display result variable
result

```
Out[5]:
```
Summary (plain English):

- What was checked: traces whose trace_name mentions any candidate service (ts-auth/order/route/train/travel) were analyzed for global P95 exceedances during the incident window 2024-01-23 21:01:34–21:31:34 UTC.

- Key finding: multiple trace groups show short-lived anomalies (value > group global P95) inside the window. The returned rows (top 20) all have window_anomaly_count = 2 (i.e., two samples above P95) and full_count ≈ 24–25.

- Timing: the earliest anomaly timestamps are clustered very early in the window — mostly at 2024-01-23T21:04:00Z and 21:05:00Z (some groups show 21:04:00 as the first exceedance, others 21:05:00). This indicates the issue began around 21:04–21:05 UTC.

- Affected trace types: duration metrics (duration_mean, duration_p95) and some row_count metrics are affected. Max values in-window exceed the computed P95 thresholds for those groups (i.e., clear spikes in latency/throughput).

- Services and call relationships involved (examples from top results):
  - Calls involving ts-travel-service and ts-route-service (many duration traces with earliest anomalies at ~21:04).
  - Traces with cmdb_id values like ts-route-service, ts-travel-service, ts-train-service, ts-order-service (and some caller services such as ts-ticketinfo, ts-inside-payment) show anomalies — meaning both callers and the candidate services are implicated via trace names that include the candidate substrings.
  - Some row_count anomalies were observed (e.g., to_ts-order-service) indicating brief changes in throughput.

- What was not seen: these are duration/row_count spikes — no evidence here of sustained error-rate spikes in the trace groups shown.

- Recommended next steps:
  1. Focus investigation on ~21:04–21:05 UTC. Inspect span details for the traces that list ts-travel-service and ts-route-service to find slow operations or downstream delays.
  2. Correlate these trace spikes with the metric and log anomalies already found (CPU/diskio/memory/latency and row_count spikes) around the same times.
  3. Examine caller services (e.g., ts-ticketinfo, ts-inside-payment) to see whether increased load or upstream delays propagated to the candidate services.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  full_count  p95_threshold  window_count  window_anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window
64            ts-route-service  trace.from_ts-travel-service.duration_mean          25       0.002966            25                     2      2024-01-23 21:04:00+00:00             0.003015
32             ts-food-service    trace.to_ts-travel-service.duration_mean          25       0.010063            25                     2      2024-01-23 21:04:00+00:00             0.010582
116          ts-travel-service     trace.to_ts-route-service.duration_mean          25       0.002966            25                     2      2024-01-23 21:04:00+00:00             0.003015
61            ts-route-service    trace.from_ts-route-service.duration_p95          25       0.001314            25                     2      2024-01-23 21:04:00+00:00             0.001339
69            ts-route-service      trace.to_ts-route-service.duration_p95          25       0.001314            25                     2      2024-01-23 21:04:00+00:00             0.001339
128         ts-travel2-service     trace.to_ts-route-service.duration_mean          25       0.002840            25                     2      2024-01-23 21:04:00+00:00             0.002940
104           ts-train-service     trace.to_ts-train-service.duration_mean          25       0.000810            25                     2      2024-01-23 21:05:00+00:00             0.000812
93       ts-ticketinfo-service   trace.from_ts-travel-service.duration_p95          25       0.048232            25                     2      2024-01-23 21:05:00+00:00             0.048296
45            ts-order-service   trace.from_ts-travel-service.duration_p95          25       0.014620            25                     2      2024-01-23 21:05:00+00:00             0.050509
44            ts-order-service  trace.from_ts-travel-service.duration_mean          25       0.006506            25                     2      2024-01-23 21:05:00+00:00             0.010423
85         ts-security-service      trace.to_ts-order-service.duration_p95          25       0.046276            25                     2      2024-01-23 21:05:00+00:00             0.055746
92       ts-ticketinfo-service  trace.from_ts-travel-service.duration_mean          25       0.017148            25                     2      2024-01-23 21:05:00+00:00             0.017463
39   ts-inside-payment-service         trace.to_ts-order-service.row_count          24       9.850000            24                     2      2024-01-23 21:05:00+00:00            10.000000
37   ts-inside-payment-service      trace.to_ts-order-service.duration_p95          24       0.006969            24                     2      2024-01-23 21:05:00+00:00             0.024592
36   ts-inside-payment-service     trace.to_ts-order-service.duration_mean          24       0.004900            24                     2      2024-01-23 21:05:00+00:00             0.008023
108          ts-travel-service  trace.from_ts-travel-service.duration_mean          25       0.010211            25                     2      2024-01-23 21:05:00+00:00             0.010916
57         ts-preserve-service     trace.to_ts-travel-service.duration_p95          25       0.446493            25                     2      2024-01-23 21:05:00+00:00             0.557483
109          ts-travel-service   trace.from_ts-travel-service.duration_p95          25       0.046862            25                     2      2024-01-23 21:05:00+00:00             0.048359
97            ts-train-service    trace.from_ts-train-service.duration_p95          25       0.001162            25                     2      2024-01-23 21:05:00+00:00             0.001182
105           ts-train-service      trace.to_ts-train-service.duration_p95          25       0.001162            25                     2      2024-01-23 21:05:00+00:00             0.001182```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import re

# Reuse existing trace_df if available; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Parse timestamp as UTC datetimes and ensure numeric values
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Extract target service when trace_name contains 'to_ts-<service>'
trace_df['target_service'] = trace_df['trace_name'].astype(str).str.extract(r'to_ts-([A-Za-z0-9\-]+)', expand=False)

# Keep only rows where extracted target is in candidates
to_target_df = trace_df[trace_df['target_service'].isin(candidates)].copy()

if to_target_df.empty:
    result = "Empty result: no 'to_ts-' target rows exist for the candidate services."
else:
    # Compute full-series stats per (target_service, trace_name) BEFORE window filtering
    group_stats = (
        to_target_df
        .groupby(['target_service','trace_name'], as_index=False)['value']
        .agg(full_count='size', p95_threshold=lambda s: float(s.quantile(0.95)))
    )

    # Incident window (UTC)
    window_start = pd.to_datetime('2024-01-23 21:01:34', utc=True)
    window_end   = pd.to_datetime('2024-01-23 21:31:34', utc=True)

    # Rows within the window
    window_df = to_target_df[(to_target_df['timestamp'] >= window_start) & (to_target_df['timestamp'] <= window_end)].copy()

    if window_df.empty:
        result = "Empty result: matched 'to_ts-' rows exist but none fall inside the incident window."
    else:
        # Merge p95 into window rows and flag anomalies
        window_merged = window_df.merge(group_stats[['target_service','trace_name','p95_threshold']],
                                        on=['target_service','trace_name'], how='left')
        window_merged['is_anomaly'] = window_merged['value'] > window_merged['p95_threshold']

        # Aggregates in window per (target_service, trace_name)
        window_counts = (
            window_merged
            .groupby(['target_service','trace_name'], as_index=False)
            .agg(window_count=('value','size'),
                 window_anomaly_count=('is_anomaly','sum'),
                 max_value_in_window=('value','max'))
        )

        # Earliest anomaly timestamp per group (if any)
        anomalies = window_merged[window_merged['is_anomaly']]
        if not anomalies.empty:
            earliest_anom = (
                anomalies
                .groupby(['target_service','trace_name'], as_index=False)['timestamp'].min()
            )
            earliest_anom['earliest_anomaly_timestamp_utc'] = earliest_anom['timestamp'].dt.tz_convert('UTC').astype(str)
            earliest_anom = earliest_anom.drop(columns=['timestamp'])
        else:
            earliest_anom = pd.DataFrame(columns=['target_service','trace_name','earliest_anomaly_timestamp_utc'])

        # Combine group-level stats
        groups_combined = group_stats.merge(window_counts, on=['target_service','trace_name'], how='left')
        groups_combined = groups_combined.merge(earliest_anom, on=['target_service','trace_name'], how='left')

        # Fill missing window aggregates
        groups_combined['window_count'] = groups_combined['window_count'].fillna(0).astype(int)
        groups_combined['window_anomaly_count'] = groups_combined['window_anomaly_count'].fillna(0).astype(int)
        groups_combined['max_value_in_window'] = groups_combined['max_value_in_window'].where(groups_combined['max_value_in_window'].notna(), pd.NA)
        groups_combined['earliest_anomaly_timestamp_utc'] = groups_combined['earliest_anomaly_timestamp_utc'].where(
            groups_combined['earliest_anomaly_timestamp_utc'].notna(), None
        )

        # Keep only groups with anomalies
        anomalous_groups = groups_combined[groups_combined['window_anomaly_count'] > 0].copy()
        if anomalous_groups.empty:
            result = "Empty result: no target trace groups have anomalies (value > global P95) in the incident window."
        else:
            # Compute breach ratio per group (handle p95==0)
            def compute_breach_ratio(row):
                p95 = row['p95_threshold']
                maxv = row['max_value_in_window']
                if pd.isna(maxv):
                    return 0.0
                if p95 == 0:
                    return float('inf') if maxv > 0 else 0.0
                return float(maxv) / float(p95)

            anomalous_groups['breach_ratio'] = anomalous_groups.apply(compute_breach_ratio, axis=1)

            # Aggregate per target_service
            agg = (
                anomalous_groups
                .groupby('target_service', as_index=False)
                .agg(
                    num_trace_groups_with_anomaly=('trace_name','nunique'),
                    earliest_anomaly_timestamp_utc=('earliest_anomaly_timestamp_utc','min'),
                    max_breach_ratio=('breach_ratio','max')
                )
            )

            # Prepare top groups per target (up to 5), sorted by earliest anomaly timestamp asc
            anomalous_groups_sorted = anomalous_groups.sort_values(['target_service','earliest_anomaly_timestamp_utc'], na_position='last')
            top_groups = (
                anomalous_groups_sorted
                .groupby('target_service', group_keys=False)
                .apply(lambda d: d.sort_values('earliest_anomaly_timestamp_utc').head(5))
                .reset_index(drop=True)
            )

            top_groups_df = top_groups[['target_service','trace_name','earliest_anomaly_timestamp_utc','p95_threshold','max_value_in_window']].copy()

            # Final summary: join agg with earliest timestamp for sorting, limit to top 5 targets sorted by earliest anomaly ascending
            agg_sorted = agg.sort_values('earliest_anomaly_timestamp_utc', na_position='last').head(5).reset_index(drop=True)

            # Keep only targets with num_trace_groups_with_anomaly > 0 (already the case)
            summary_df = agg_sorted[['target_service','num_trace_groups_with_anomaly','earliest_anomaly_timestamp_utc','max_breach_ratio']].copy()

            # Compact final result: summary_df and top_groups_df
            result_summary = summary_df
            result_top_groups = top_groups_df

            result = (result_summary, result_top_groups)

# Display the result variable(s)
result

```
Out[7]:
```
Summary (plain English):

- I searched trace_name values for patterns of the form "to_ts-<service>" and kept only those targets that are in the candidate list (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service).

- Result: no trace rows matched that "to_ts-<candidate>" pattern. Therefore there are no per-target aggregates or anomaly results to report for these "to_ts-" targets within the incident window.

- Next step suggestion: analyze other trace_name patterns that reference candidate services (e.g., "from_ts-..." or any trace_name containing the candidate substrings), since earlier analyses found anomalies when matching broader trace_name patterns.

The original code execution output of IPython Kernel is also provided below for reference:

Empty result: no 'to_ts-' target rows exist for the candidate services.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamps parsed as UTC and numeric values
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Restrict to rows where cmdb_id is one of the candidates
cand_traces = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

if cand_traces.empty:
    result = "Empty result: none of the candidate services have trace rows in trace.csv"
else:
    # Compute full-series group stats (full_count and global P95) BEFORE window filtering
    group_stats = (
        cand_traces
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(full_count='size', p95_threshold=lambda s: float(s.quantile(0.95)))
    )

    # Incident window (UTC inclusive)
    window_start = pd.to_datetime('2024-01-23 21:01:34', utc=True)
    window_end   = pd.to_datetime('2024-01-23 21:31:34', utc=True)

    # Rows within the window
    window_df = cand_traces[(cand_traces['timestamp'] >= window_start) & (cand_traces['timestamp'] <= window_end)].copy()

    if window_df.empty:
        result = "Empty result: candidate trace rows exist but none fall inside the incident window."
    else:
        # Merge p95 into window rows and flag anomalies
        window_merged = window_df.merge(group_stats[['cmdb_id','trace_name','p95_threshold']],
                                        on=['cmdb_id','trace_name'], how='left')
        window_merged['is_anomaly'] = window_merged['value'] > window_merged['p95_threshold']

        # Aggregates per (cmdb_id, trace_name) within window
        window_agg = (
            window_merged
            .groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(
                window_count=('value','size'),
                window_anomaly_count=('is_anomaly','sum'),
                max_value_in_window=('value','max')
            )
        )

        # Earliest anomaly timestamp per group (if any)
        anomalies = window_merged[window_merged['is_anomaly']]
        if not anomalies.empty:
            earliest = (
                anomalies
                .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
                .min()
            )
            earliest = earliest.rename(columns={'timestamp':'earliest_anomaly_timestamp_utc'})
        else:
            earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp_utc'])

        # Combine group-level info: join with full-series stats
        groups = group_stats.merge(window_agg, on=['cmdb_id','trace_name'], how='left')
        groups = groups.merge(earliest, on=['cmdb_id','trace_name'], how='left')

        # Fill NaNs for window aggregates
        groups['window_count'] = groups['window_count'].fillna(0).astype(int)
        groups['window_anomaly_count'] = groups['window_anomaly_count'].fillna(0).astype(int)
        groups['max_value_in_window'] = groups['max_value_in_window'].where(groups['max_value_in_window'].notna(), pd.NA)

        # Keep only groups that have anomalies in the window
        anomalous_groups = groups[groups['window_anomaly_count'] > 0].copy()

        if anomalous_groups.empty:
            result = "Empty result: no trace groups for candidate services have anomalies (value > global P95) in the incident window."
        else:
            # Compute breach_ratio per group (handle p95==0)
            def breach_ratio(row):
                p95 = row['p95_threshold']
                maxv = row['max_value_in_window']
                if pd.isna(maxv):
                    return 0.0
                if p95 == 0:
                    return float('inf') if maxv > 0 else 0.0
                return float(maxv) / float(p95)

            anomalous_groups['breach_ratio'] = anomalous_groups.apply(breach_ratio, axis=1)

            # Ensure earliest_anomaly_timestamp_utc is datetime for sorting/aggregation
            anomalous_groups['earliest_anomaly_ts_dt'] = pd.to_datetime(anomalous_groups['earliest_anomaly_timestamp_utc'], utc=True)

            # Aggregate per cmdb_id (candidate service)
            agg = (
                anomalous_groups
                .groupby('cmdb_id', as_index=False)
                .agg(
                    num_trace_groups_with_anomaly=('trace_name','nunique'),
                    earliest_anomaly_timestamp_utc=('earliest_anomaly_ts_dt','min'),
                    max_breach_ratio=('breach_ratio','max')
                )
            )

            # Convert earliest_anomaly_timestamp_utc back to ISO strings
            agg['earliest_anomaly_timestamp_utc'] = agg['earliest_anomaly_timestamp_utc'].dt.tz_convert('UTC').astype(str)

            # Prepare up to 5 representative trace groups per cmdb_id sorted by earliest anomaly
            rep_groups = (
                anomalous_groups
                .sort_values(['cmdb_id','earliest_anomaly_ts_dt'], na_position='last')
                .groupby('cmdb_id', group_keys=False)
                .apply(lambda d: d[['cmdb_id','trace_name','earliest_anomaly_ts_dt','p95_threshold','max_value_in_window']].head(5))
                .reset_index(drop=True)
            )
            # Rename timestamp column for display
            rep_groups = rep_groups.rename(columns={'earliest_anomaly_ts_dt':'earliest_anomaly_timestamp_utc'})
            rep_groups['earliest_anomaly_timestamp_utc'] = rep_groups['earliest_anomaly_timestamp_utc'].dt.tz_convert('UTC').astype(str)

            # Keep only candidates with anomalies (already the case) and sort top candidates by earliest anomaly ascending, limit to 5
            top_candidates = agg.sort_values('earliest_anomaly_timestamp_utc', na_position='last').head(5).reset_index(drop=True)

            # Filter rep_groups to only include the top_candidates
            top_cmbids = top_candidates['cmdb_id'].tolist()
            rep_groups_top = rep_groups[rep_groups['cmdb_id'].isin(top_cmbids)].copy()

            # Final compact outputs: top_candidates summary and rep_groups_top (both DataFrames)
            summary_df = top_candidates[['cmdb_id','num_trace_groups_with_anomaly','earliest_anomaly_timestamp_utc','max_breach_ratio']].copy()
            rep_groups_df = rep_groups_top[['cmdb_id','trace_name','earliest_anomaly_timestamp_utc','p95_threshold','max_value_in_window']].copy()

            result = summary_df, rep_groups_df

# Display the result variables (compact)
result

```
Out[8]:
```
Summary (plain English):

- Overall result: All five candidate services show trace-group anomalies during the incident window, with earliest anomalies clustered at ~21:04–21:05 UTC (ts-auth anomalies appear later around 21:09+).

- Per-target summary (only targets with anomalies):
  - ts-route-service: 15 trace groups breached P95; earliest anomaly 2024-01-23T21:04:00Z; max breach ratio ≈ 1.09 (max ≈ 9% above P95).
  - ts-travel-service: 36 trace groups breached P95; earliest anomaly 2024-01-23T21:04:00Z; max breach ratio ≈ 3.45 (some traces peaked ≈3.45× their P95).
  - ts-order-service: 25 trace groups breached P95; earliest anomaly 2024-01-23T21:05:00Z; max breach ratio ≈ 3.53 (largest relative spike ~3.5× P95).
  - ts-train-service: 15 trace groups breached P95; earliest anomaly 2024-01-23T21:05:00Z; max breach ratio ≈ 1.21.
  - ts-auth-service: 9 trace groups breached P95; earliest anomaly 2024-01-23T21:09:00Z; max breach ratio ≈ 1.07.

- Representative affected trace types (examples):
  - Row_count and duration metrics (duration_mean, duration_p95, row_count) across many caller→target traces (e.g., trace.from_ts-*, trace.to_ts-*, trace.from_root.*).
  - Example traces for ts-auth: trace.from_root.row_count, trace.from_ts-auth-service.row_count/duration_p95 (earliest ~21:09).
  - Example traces for ts-travel: many duration and row_count traces from various callers (earliest ~21:04), including some with the largest relative breaches.
  - ts-order shows large relative breaches (~3.5× P95) starting ~21:05.

- Interpretation:
  - Anomalies are widespread across tracing groups and services, concentrated around 21:04–21:05 UTC, with ts-travel and ts-order showing the largest relative spikes. ts-auth shows anomalies later and smaller in relative size.
  - These are latency/throughput spikes (durations and row_counts), not sustained error-rate increases in traces.

- Recommended next steps:
  1. Prioritize ts-travel and ts-order for deep trace/span inspection around 21:04–21:05 UTC to find slow spans or downstream delays.
  2. Correlate these trace spikes with the metric and log anomalies already observed (CPU, memory, disk I/O, and row_count spikes around the same times).
  3. Inspect caller traces (ticketinfo, inside-payment, preserve, etc.) that appear in representative groups to see whether upstream load or backpressure propagated to these targets.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  num_trace_groups_with_anomaly earliest_anomaly_timestamp_utc  max_breach_ratio
0   ts-route-service                             15      2024-01-23 21:04:00+00:00          1.091877
1  ts-travel-service                             36      2024-01-23 21:04:00+00:00          3.454905
2   ts-order-service                             25      2024-01-23 21:05:00+00:00          3.528569
3   ts-train-service                             15      2024-01-23 21:05:00+00:00          1.210405
4    ts-auth-service                              9      2024-01-23 21:09:00+00:00          1.071074,               cmdb_id                                    trace_name earliest_anomaly_timestamp_utc  p95_threshold  max_value_in_window
0     ts-auth-service                     trace.from_root.row_count      2024-01-23 21:09:00+00:00      88.600000            92.000000
1     ts-auth-service          trace.from_ts-auth-service.row_count      2024-01-23 21:09:00+00:00     356.400000           368.000000
2     ts-auth-service            trace.to_ts-auth-service.row_count      2024-01-23 21:09:00+00:00     356.400000           368.000000
3     ts-auth-service       trace.from_ts-auth-service.duration_p95      2024-01-23 21:19:00+00:00       0.208390             0.209466
4     ts-auth-service         trace.to_ts-auth-service.duration_p95      2024-01-23 21:19:00+00:00       0.208390             0.209466
..                ...                                           ...                            ...            ...                  ...
20  ts-travel-service      trace.from_ts-food-service.duration_mean      2024-01-23 21:04:00+00:00       0.010063             0.010582
21  ts-travel-service       trace.to_ts-route-service.duration_mean      2024-01-23 21:04:00+00:00       0.002966             0.003015
22  ts-travel-service                  trace.from_root.duration_p95      2024-01-23 21:05:00+00:00       0.949874             1.016416
23  ts-travel-service  trace.from_ts-admin-travel-service.row_count      2024-01-23 21:05:00+00:00      11.800000            12.000000
24  ts-travel-service  trace.from_ts-preserve-service.duration_mean      2024-01-23 21:05:00+00:00       0.300066             0.389759

[25 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```